# Dia 21 — O dossiê 🗂️

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-21-distro-e-dossie](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-21-distro-e-dossie)

## 🎬 Missão

> Sexta-feira, 18h40. A sala do Núcleo Coruja está vazia, exceto pela Mestra
> Iara, que pela primeira vez em três semanas está sentada esperando *você*.
> Na mesa dela, um envelope lacrado com cera azul.
>
> — Vinte pistas, recruta. Cada uma abriu a seguinte. Se todas estiverem
> certas, o lacre abre. Se uma estiver errada, ele continua fechado — e você
> sabe por onde recomeçar.
>
> Bento aparece na porta, de mochila.
> — E depois disso você escolhe a sua distro. Ninguém investiga de verdade em
> máquina emprestada.

**Sua tarefa:** reunir as 20 flags anteriores, abrir o selo com
`montar_dossie.sh` e entregar o dossiê. De quebra, escolher a distribuição
Linux que vai usar daqui pra frente.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### Que distribuição eu estou usando?

```text
"Distribuição" (distro) é o pacote completo: kernel Linux + programas +
gerenciador de pacotes + aparência. /etc/os-release é o RG da distro.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-21-distro-e-dossie" && source "$L4O_HOME/lib/comum.sh"
titulo "RG da distro"
cat /etc/os-release
echo
echo "Kernel: $(uname -r)  ·  Arquitetura: $(uname -m)"

### Qual família? (descubra pelo gerenciador de pacotes)

```text
As distros se agrupam em famílias. A forma mais rápida de saber a família
é ver qual gerenciador de pacotes existe:
  apt     -> Debian, Ubuntu, Kali, Parrot, Tails
  dnf     -> Fedora, RHEL, Rocky
  pacman  -> Arch, Manjaro, BlackArch
  zypper  -> openSUSE
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-21-distro-e-dossie" && source "$L4O_HOME/lib/comum.sh"
titulo "Família pelo gerenciador de pacotes"
for gerenciador in apt dnf pacman zypper apk; do
  if command -v "$gerenciador" >/dev/null 2>&1; then
    ok "$gerenciador encontrado"
  else
    printf '  · %s ausente\n' "$gerenciador"
  fi
done

### A variável ID_LIKE diz quem é o "pai"

```text
Muitas distros declaram de quem derivam. Ubuntu diz "debian"; Kali também.
source carrega o arquivo como variáveis do Bash (ele é escrito nesse formato).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-21-distro-e-dossie" && source "$L4O_HOME/lib/comum.sh"
titulo "De quem esta distro deriva?"
(
  # shellcheck disable=SC1091
  . /etc/os-release
  echo "Distro: ${NAME:-?}  ·  Deriva de: ${ID_LIKE:-(nenhuma — é uma distro-mãe)}"
)

### Quantos programas eu instalei nestes 21 dias?

```text
dpkg -l lista todos os pacotes .deb instalados (você viu no Dia 3).
wc -l conta linhas. As 5 primeiras linhas são cabeçalho.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-21-distro-e-dossie" && source "$L4O_HOME/lib/comum.sh"
titulo "Pacotes instalados"
if command -v dpkg >/dev/null 2>&1; then
  echo "$(dpkg -l | tail -n +6 | wc -l) pacotes .deb instalados nesta máquina"
else
  alerta "Esta distro não usa dpkg."
fi

### Seu progresso até aqui

```text
Cada checar.sh que deu certo guardou a flag em .progresso/ (pasta ignorada
pelo git). Vamos contar quantas você tem.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-21-distro-e-dossie" && source "$L4O_HOME/lib/comum.sh"
titulo "Pistas no seu dossiê"
total="$(find "$L4O_HOME/.progresso" -name '*.flag' 2>/dev/null | wc -l)"
echo "Você tem $total de 20 pistas registradas."
if [ "$total" -lt 20 ]; then
  dica "Volte aos dias que faltam e rode: bash checar.sh 'VAZANTE{...}'"
fi

### Como funciona o selo (versão de brinquedo)

```text
O selo final usa duas ideias que você já conhece de nome:
  1. HASH: sha256sum transforma qualquer texto num "resumo" de 64 letras.
     Mudou uma vírgula, muda tudo.
  2. XOR: combinar bytes com uma chave. Fazer XOR duas vezes com a mesma
     chave devolve o original — cifra e decifra com o mesmo gesto.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-21-distro-e-dossie" && source "$L4O_HOME/lib/comum.sh"
titulo "Hash: uma vírgula muda tudo"
printf '%s' "maré alta" | sha256sum
printf '%s' "maré alta." | sha256sum

titulo "XOR: cifrar e decifrar com a mesma chave"
python3 - <<'PY'
chave = b"coruja"
mensagem = b"pista secreta"
cifrado = bytes(b ^ chave[i % len(chave)] for i, b in enumerate(mensagem))
decifrado = bytes(b ^ chave[i % len(chave)] for i, b in enumerate(cifrado))
print("cifrado  :", cifrado.hex())
print("decifrado:", decifrado.decode())
PY

### Montando o dossiê

```text
O montar_dossie.sh confere as 20 flags e, se estiverem todas certas, usa o
SHA-256 delas como chave para abrir o selo. Sem as 20, o selo não abre.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-21-distro-e-dossie" && source "$L4O_HOME/lib/comum.sh"
titulo "Tentando abrir o selo"
bash montar_dossie.sh || dica "Ainda faltam pistas. Tudo bem — o dossiê espera por você."

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-21-distro-e-dossie" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-21-distro-e-dossie" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)